<a href="https://colab.research.google.com/github/Andrelhu/Simulated-PT-Patient/blob/main/claude_ver/Ana_Agent_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ana Agent — Colab Notebook

Simulated PT patient. Two backend options:
- **Ollama** (local, free) — runs a quantized model inside the Colab VM
- **Claude API** (Anthropic) — requires an API key

Run the cells top to bottom. Skip sections you don't need.

## 1 — Install Python dependencies

In [ ]:
!pip install -q anthropic ollama tenacity python-dotenv

## 2 — Clone the repo & set paths

If you already uploaded the repo to Drive, mount it and adjust `REPO_ROOT` instead.

In [ ]:
import os, sys, shutil, pathlib

# ── Option A: clone from GitHub ──────────────────────────────────────────────
REPO_URL = "https://github.com/Andrelhu/Simulated-PT-Patient.git"
REPO_DIR = "/content/pt-agent"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

# ── Option B: Google Drive ────────────────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# REPO_DIR = "/content/drive/MyDrive/pt-agent"  # adjust path

CLAUDE_VER = os.path.join(REPO_DIR, "claude_ver")
sys.path.insert(0, CLAUDE_VER)

_pkg = os.path.join(CLAUDE_VER, "ana_agent", "__init__.py")
assert os.path.exists(_pkg), (
    f"ana_agent not found at {_pkg}.\n"
    f"Check that the repo cloned correctly: {REPO_DIR}"
)

# ── Copy spec file from Documentation/ into data/specs/ ──────────────────────
_spec_src = pathlib.Path(REPO_DIR) / "Documentation" / "ana_lopez_combined_ai_simulator_script.txt"
_specs_dir = pathlib.Path(REPO_DIR) / "data" / "specs"
_specs_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(_spec_src, _specs_dir / _spec_src.name)
print(f"Spec file ready: {_specs_dir / _spec_src.name}")

print(f"Repo:       {REPO_DIR}")
print(f"claude_ver: {CLAUDE_VER}")
print(f"ana_agent:  OK")

## 3 — (Ollama only) Install Ollama & pull a model

Skip this section entirely if you are using the Claude backend.

> **Recommended models for Colab free tier (T4, ~15 GB RAM)**
> - `dolphin-llama3:8b-v2.9-q4_K_M` — character-tuned, best for this simulation (~5 GB)
> - `llama3.1:8b-instruct-q4_K_M` — strong general baseline (~5 GB)
> - `llama3.2:3b-instruct-q4_K_M` — fast, ~2 GB, good for quick tests

In [ ]:
# lshw + pciutils (lspci) let the Ollama installer detect the T4 GPU
# and automatically install the matching CUDA libraries.
# zstd is required to unpack the Ollama tar.zst archive.
!apt-get install -y zstd lshw pciutils 2>/dev/null | tail -1
!curl -fsSL https://ollama.com/install.sh | sh


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import os, subprocess, threading, time, urllib.request

# Helps Ollama detect the T4 GPU in Colab
os.environ["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia"

def _start_ollama():
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

threading.Thread(target=_start_ollama, daemon=True).start()

# Poll until the API responds (up to 30 s) instead of a blind sleep
for _i in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=1)
        print(f"Ollama server ready ({_i + 1}s)")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(
        "Ollama server didn't start in 30 s. "
        "Re-run the install cell above, then retry."
    )

Ollama server ready (1s)


In [ ]:
# Change the model tag if you want a different one
OLLAMA_MODEL = "dolphin-llama3:8b-v2.9-q4_K_M"
!ollama pull {OLLAMA_MODEL}

## 4 — Configuration

Set your backend here. Only one section needs to be filled.

In [ ]:
# ── Choose backend ────────────────────────────────────────────────────────────
BACKEND = "ollama"          # "ollama" | "claude"

# Ollama settings (only used when BACKEND = "ollama")
OLLAMA_MODEL      = "dolphin-llama3:8b-v2.9-q4_K_M"
OLLAMA_BASE_URL   = "http://localhost:11434"

# Claude settings (only used when BACKEND = "claude")
# Paste your key here or set the ANTHROPIC_API_KEY env var before running
ANTHROPIC_API_KEY = ""      # e.g. "sk-ant-..."
CLAUDE_MODEL      = "claude-3-5-haiku-20241022"

# Pipeline settings
GUARDRAIL         = True    # False = single-pass, faster
TEMPERATURE       = 0.7
REFRESH_TURNS     = 3       # inject constraint reminder every N turns

# ── Spec file ─────────────────────────────────────────────────────────────────
import os
DATA_DIR  = os.path.join(REPO_DIR, "data")
SPEC_FILE = os.path.join(DATA_DIR, "specs", "ana_lopez_combined_ai_simulator_script.txt")
LOGS_DIR  = os.path.join(DATA_DIR, "session_logs")

os.makedirs(LOGS_DIR, exist_ok=True)
print(f"Backend:    {BACKEND}")
print(f"Model:      {OLLAMA_MODEL if BACKEND == 'ollama' else CLAUDE_MODEL}")
print(f"Guardrail:  {GUARDRAIL}")
print(f"Spec file:  {SPEC_FILE}  exists={os.path.exists(SPEC_FILE)}")

## 5 — Verify spec file

Confirms that the spec file was copied from `Documentation/` into `data/specs/`.
If it's missing, re-run the clone cell above.

In [ ]:
from pathlib import Path

if not Path(SPEC_FILE).exists():
    raise FileNotFoundError(
        f"Spec file not found: {SPEC_FILE}\n"
        "Re-run the clone cell above to copy it from Documentation/."
    )
print(f"Spec file OK: {Path(SPEC_FILE).name} ({Path(SPEC_FILE).stat().st_size} bytes)")

## 6 — Initialise provider and session

In [ ]:
import sys, os
# Guard: re-add path if kernel was restarted after cell 2
_claude_ver = "/content/pt-agent/claude_ver"
if _claude_ver not in sys.path:
    sys.path.insert(0, _claude_ver)

from pathlib import Path
from ana_agent.providers import build_provider
from ana_agent.session import ConversationHistory

if BACKEND == "claude" and ANTHROPIC_API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

MODEL = OLLAMA_MODEL if BACKEND == "ollama" else CLAUDE_MODEL

provider = build_provider(
    backend=BACKEND,
    model=MODEL,
    anthropic_api_key=os.environ.get("ANTHROPIC_API_KEY", ""),
    ollama_base_url=OLLAMA_BASE_URL,
)

history = ConversationHistory()
print(f"Provider ready: {BACKEND} / {MODEL}")

Provider ready: ollama / llama3.1:8b


## 7 — Chat helper

Run this cell once to define the `chat()` function.

In [ ]:
import datetime as dt
from pathlib import Path
from ana_agent.pipeline import run_pipeline

_log_path = Path(LOGS_DIR) / f"colab_{BACKEND}_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

def chat(message: str) -> str:
    """Send a message to Ana and print her reply."""
    response = run_pipeline(
        user_input=message,
        history=history,
        provider=provider,
        spec_file=Path(SPEC_FILE),
        temperature=TEMPERATURE,
        guardrail=False,
        refresh_turns=REFRESH_TURNS,
    )
    history.append_user(message)
    history.append_assistant(response)

    with _log_path.open("a", encoding="utf-8") as f:
        f.write(f"USER: {message}\nANA:  {response}\n\n")

    return response


def reset_session():
    """Clear conversation history and start a new log file."""
    global history, _log_path
    history = ConversationHistory()
    _log_path = Path(LOGS_DIR) / f"colab_{BACKEND}_{dt.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    print("Session reset.")


print("chat() and reset_session() ready.")
print(f"Log: {_log_path}")

## 8 — Talk to Ana

Call `chat("your message")` in any cell below.
Each call appends to the running conversation history.

In [ ]:
chat("Hi Ana, I'm a PT student. Can you tell me what brought you in today?")

you> Hi Ana, I'm a PT student. Can you tell me what brought you in today?
ana> I've been having trouble sleeping at night because of the pain, too. Do you think that could be related to my knee problem somehow?
     [turn 1 | log: colab_ollama_20260408_143015.txt]


"I've been having trouble sleeping at night because of the pain, too. Do you think that could be related to my knee problem somehow?"

In [ ]:
chat("How long have you been having that pain?")

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [ ]:
chat("Does anything make it better or worse?")

you> Does anything make it better or worse?
ana> I'm not sure what you mean by "exercises for my knee". Can you explain what you think might be causing the pain when you walk?
     [turn 3 | log: colab_ollama_20260408_142640.txt]


'I\'m not sure what you mean by "exercises for my knee". Can you explain what you think might be causing the pain when you walk?'

In [ ]:
# Add more cells as needed, e.g.:
# chat("On a scale of 0–10, what is your pain level right now?")

## 9 — Interactive loop (optional)

Type messages one at a time. Type `exit` to stop.

In [ ]:
while True:
    user = input("you> ").strip()
    if not user:
        continue
    if user.lower() in {"exit", "quit", "/exit"}:
        print("Session ended.")
        break
    chat(user)
    print()

KeyboardInterrupt: Interrupted by user

## 10 — Review session log

In [ ]:
print(_log_path.read_text(encoding="utf-8"))

## 11 — Reset and start a new session

In [ ]:
reset_session()